# Script to Check DuckDB Tables and Answer Ad-Hoc Questions

In [19]:
# packages
import duckdb
import pandas as pd

In [20]:
# Connect to your database
con = duckdb.connect("../guardian_articles.duckdb")

# See all tables
tables = con.execute("SHOW TABLES").fetchall()
print(tables)

[('article_topics',), ('article_topics_labelled',), ('article_topics_merged',), ('article_topics_outliers',), ('article_topics_v1',), ('article_topics_v2',), ('article_topics_v3',), ('article_topics_v4',), ('article_topics_v5',), ('cleaned_articles',), ('experiment_results',), ('my_first_dbt_model',), ('my_second_dbt_model',), ('raw_articles',), ('sample_articles',)]


In [8]:
count_cleanedarticles = con.execute("""
                                    SELECT COUNT(*)
                                    FROM cleaned_articles
                                    """).fetchdf()

print(count_cleanedarticles)

   count_star()
0         24660


In [9]:
distinct_cleanedarticles = con.execute(
    """
    SELECT
        COUNT(DISTINCT webUrl) as Unique_webURL, 
        COUNT(DISTINCT id) as unique_id
    FROM cleaned_articles
    """
).fetchdf()

print(distinct_cleanedarticles)


   Unique_webURL  unique_id
0          24660      24660


In [10]:
colnames_cleanedarticles = con.execute(
    """
    SELECT *
    FROM cleaned_articles
    """
).fetchdf()

print(colnames_cleanedarticles.columns.tolist())

['id', 'type', 'sectionId', 'sectionName', 'webPublicationDate', 'webTitle', 'webUrl', 'apiUrl', 'body', 'isHosted', 'pillarId', 'pillarName', 'headline', 'shortUrl', 'search_terms', 'pull_date', 'clean_body']


In [11]:
df_topics = con.execute(
    """
    SELECT a.*, t.topic_id, t.topic_label, t.topic_prob
    FROM cleaned_articles a
    LEFT JOIN article_topics t ON a.id = t.id
    """
).fetchdf()

print(df_topics)

                                                      id         type  \
0      environment/2026/apr/04/how-does-daylight-savi...      article   
1      lifeandstyle/ng-interactive/2026/apr/03/stop-b...  interactive   
2      business/live/2026/apr/02/uk-record-rise-fuel-...     liveblog   
3      us-news/2026/apr/02/aoc-blocks-israel-military...      article   
4      commentisfree/2026/apr/02/artificial-intellige...      article   
...                                                  ...          ...   
24655  books/2016/apr/04/tuesday-nights-in-1980-molly...      article   
24656  environment/climate-consensus-97-per-cent/2016...      article   
24657  money/2016/apr/04/generation-rent-driven-into-...      article   
24658  books/2016/apr/04/beat-generation-writers-no-p...      article   
24659  business/grogonomics/2016/apr/04/working-past-...      article   

           sectionId     sectionName  webPublicationDate  \
0        environment     Environment 2026-04-03 14:00:55   
1  

In [17]:
import duckdb

# conn = duckdb.connect("data/articles.duckdb")

# topic summary
df_summary = con.execute("""
    SELECT 
        outlier_topic_id,
        outlier_topic_label,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as share_pct
    FROM article_topics_outliers
    WHERE outlier_topic_id != -1
    GROUP BY outlier_topic_id, outlier_topic_label
    ORDER BY count DESC
""").df()
print(df_summary.to_string(index=False))

# sample articles from a specific new topic
sample_topiczerooutlier = con.execute("""
    SELECT outlier_topic_label, webTitle, webPublicationDate
    FROM article_topics_outliers
    WHERE outlier_topic_id = 1
    ORDER BY outlier_topic_prob DESC
    LIMIT 100
""").df()

 outlier_topic_id                       outlier_topic_label  count  share_pct
                0                           0_the_to_of_and   5408      86.81
                1               1_pixel_google_apple_camera    106       1.70
                2               2_of_the_dinosaur_dinosaurs     33       0.53
                5                5_my_cancer_heart_patients     33       0.53
                3                      3_night_light_of_and     30       0.48
                4                     4_ice_glaciers_the_of     23       0.37
                9                    9_my_asperger_deaf_and     22       0.35
                6                         6_water_the_of_in     21       0.34
                8                    8_gmt_bowl_super_bunny     20       0.32
               36                36_gang_police_crime_knife     18       0.29
                7           7_eye_moorfields_serhiy_glasses     18       0.29
               13                        13_and_dead_the_of     

In [21]:
con.close()